Hacer dos dashboards en los que se muestren gráficos estáticos y dinámicos de diferentes tipos sobre los datos. Debe de haber gráficas de muchos tipos, también algún mapa. Las columnas son:
title: title name given to the earthquake
magnitude: The magnitude of the earthquake
date_time: date and time
cdi: The maximum reported intensity for the event range
mmi: The maximum estimated instrumental intensity for the event
alert: The alert level - “green”, “yellow”, “orange”, and “red”
tsunami: "1" for events in oceanic regions and "0" otherwise
sig: A number describing how significant the event is. Larger numbers indicate a more significant event. This value is determined on a number of factors, including: magnitude, maximum MMI, felt reports, and estimated impact
net: The ID of a data contributor. Identifies the network considered to be the preferred source of information for this event.
nst: The total number of seismic stations used to determine earthquake location.
dmin: Horizontal distance from the epicenter to the nearest station
gap: The largest azimuthal gap between azimuthally adjacent stations (in degrees). In general, the smaller this number, the more reliable is the calculated horizontal position of the earthquake. Earthquake locations in which the azimuthal gap exceeds 180 degrees typically have large location and depth uncertainties
magType: The method or algorithm used to calculate the preferred magnitude for the event
depth: The depth where the earthquake begins to rupture
latitude / longitude: coordinate system by means of which the position or location of any place on Earth's surface can be determined and described
location: location within the country
continent: continent of the earthquake hit country
country: affected country

Algunas librerías que se podrían usar son:
plotly
plotly.express
dash
panel
streamlit
geoplot
cartopy
folium
branca
matplotlib
seaborn
hvplot.pandas
mpl_toolkits.basemap
pywaffle
pypalettes
pyfonts
highlight_text
drawarrow
pandas
geopandas
mapclassify
geodatasets
numpy
netCDF4

3 dashboards con temáticas diferentes:

espacio

barplot: Número de terremotos por title, continent, country, location o alert. Visualización cambiable con dropdown menu. There is also a radio button that changes the plot between a barplot and a wordcloud
treemap: Número de terremotos por continent → country → location
dendogram: Agrupamiento jerárquico por latitude, longitude, depth, magnitude.
area chart where the y axis is latitude, being the y=0 the equator
map: Scatter geográfico estilo latitude vs longitude, tamaño por magnitude, color por alert. Los tsunamis son círculos en vez de cuadrados
cloropleth: número de terremotos, media de magnitud, mayor maginutd, media de sig (significative), número de tsunamis, interpolación entre valores de alert para conseguir uno medio por país, media de mmi.
box plots: Distribución de magnitude por continent.
also make, with an ai, a prediction of where the next earthquake will be and its magnite: Entrenar un modelo simple de regresión o un Random Forest usando: Inputs: latitude, longitude, depth, magnitude, date_time, country, continent. Outputs: next probable latitude, longitude, magnitude.

tiempo

line chart: Magnitud promedio a lo largo del tiempo (date_time). Con un radio button se convierte en lollipop plot
candlestick: Magnitud mínima, máxima, apertura y cierre por día/mes.
animation: Movimiento de epicentros en el tiempo, con tamaño por magnitud y color por alerta. Utilizar un slider donde cada "frame de la animación" es un año
Time series de alerta: Evolución de alertas por mes/año. Gráfico de líneas apiladas (stacked area chart) para ver tendencias.
Calendar heatmap: Cantidad de terremotos por día o mes (intercambiable con dropdown) del año → patrón estacional.
Polar chart (circular): ver distribución de frecuencia por hora del día o mes. Ejemplo: cantidad de terremotos por hora del día → forma circular. Alternativamente: frecuencia mensual → detectar estacionalidad visual
3d plot: plot earthquakes in an earth globe and make lines so it goes from the surface until its depth. magnitude changes the width of this line. alert determines its color. this last one can be changed by the user so the depth determines the color. Also the user can filter only tsunamis. También mostrar intensidad como una capa de calor sobre el globo (en vez de solo líneas). Color: densidad de terremotos o magnitud media. Visualmente guay si se combina con transparencia de océanos y rotación automática. Para este plot el usuario puede elegir con un rango de años los años de los terremtoso que se incluyen en el plot. Obviamente el mapa de color también cambiaría.

ninguno

violin plots: Distribución de magnitude por alert.
heatmap: correlaciones entre variables
bubble: los ejes son magnitud y profundidad, el color y el tamaño la alerta
density: distribution plot of magnitude and depth intercheangeable with a radio buttons
venn diagram: Eventos con tsunami (tsunami=1) vs alerta roja (alert="red") vs magnitud>6.
pie chart: Distribución de alertas (alert) y magtype. radio buttons. También se puede cambiar de pie chart a waffle chart con un dropdown, radio button o lo que sea
radar chart: para los top 10 terremotos de la historia. Se elige terremto con un dropdown, slider o lo que creas que es mejor

In [ ]:
import datetime
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from scipy.stats import gaussian_kde
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
import io
import base64
from pywaffle import Waffle
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from googletrans import Translator


# https://www.kaggle.com/datasets/warcoder/earthquake-dataset/data
CARPETA_DATASETS = "dataset"
DATASET = "earthquake_1995-2023.csv"
RENDERER = "browser"  # browser o notebook
VALOR_SUSTITUTO_NULO = "desconocido"
COLOR_VALOR_SUSTITUTO_NULO = "gray"
# earthquake_1995-2023.csv tiene los datos de earthquake_data.csv más algunos adicionales

# ----------------------------
# 1. Cargar y limpiar dataset
# ----------------------------
df = pd.read_csv(os.path.join(CARPETA_DATASETS, "earthquake_1995-2023.csv"))

print(df.head(), "\n")

total_filas = df.shape[0]

nulos = df.isnull().sum()
nas = df.isna().sum()
vacios = (df == "").sum()

# Filtrar solo columnas con algún valor > 0
mask = (nulos > 0) | (nas > 0) | (vacios > 0)
conteos = pd.DataFrame({
    "Nulos": nulos[mask],
    "NA": nas[mask],
    "Vacíos": vacios[mask]
})

porcentajes = pd.DataFrame({
    "Nulos (%)": (nulos[mask] / total_filas * 100),
    "NA (%)": (nas[mask] / total_filas * 100),
    "Vacíos (%)": (vacios[mask] / total_filas * 100)
})

print(
    f"Número de filas: {total_filas}\n"
    f"Número de columnas: {df.shape[1]}\n\n"
    f"Conteos de valores por columna:\n{conteos}\n\n"
    f"Porcentaje de valores por columna:\n{porcentajes}"
)

# alert tiene 551 valores nulos/NA
# continent tiene 716 valores nulos/NA
# country tiene 349 valores nulos/NA
# location tiene 6 valores nulos/NA
for col in ["alert", "continent", "country", "location"]:
    df[col] = df[col].fillna(VALOR_SUSTITUTO_NULO)

traduccion_colores = {
    "green": "verde",
    "yellow": "amarillo",
    "red": "rojo",
    "orange": "naranja",
    VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
}

df["alert"] = df["alert"].map(traduccion_colores)

colores_alerta = {v: k for k, v in traduccion_colores.items()}
colores_alerta[VALOR_SUSTITUTO_NULO] = COLOR_VALOR_SUSTITUTO_NULO

posiciones_alerta = {
    "verde": 0,
    "amarillo": 1,
    "naranja": 2,
    "rojo": 3,
    VALOR_SUSTITUTO_NULO: 4
}

# traducir title?, alert, net?, magType??, location, continent, country

translator = Translator()

#for col in ["location", "country", "continent"]: # TODO fix
#    df[col] = df[col].astype(str).apply(lambda x: translator.translate(x, src='en', dest='es').text)

df["date_time"] = pd.to_datetime(df["date_time"], format="%d-%m-%Y %H:%M")
df['year'] = df['date_time'].dt.year
CONTINENTS = sorted([c for c in df['continent'].unique() if c != VALOR_SUSTITUTO_NULO]) + [VALOR_SUSTITUTO_NULO]
COUNTRIES = sorted([c for c in df['country'].unique() if c != VALOR_SUSTITUTO_NULO]) + [VALOR_SUSTITUTO_NULO]
ALERTS = sorted([c for c in df['alert'].unique()], key=lambda x: posiciones_alerta[x])
YEARS = sorted([int(y) for y in df['year'].unique()])
YEAR_MIN, YEAR_MAX = min(YEARS), max(YEARS)

"""
# ----------------------------
# 2. MAPA INTERACTIVO CON FOLIUM
# ----------------------------
map_center = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=2, tiles="cartodb positron")

# TODO if you scroll too far horizontally, the points do not appear
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["magnitude"] ** 3 / 50,  # ** 3 / 50 o ** 2 / 10 van bien
        color="crimson" if row["alert"] == "red" else "orange" if row["alert"] == "yellow" else "blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"<b>{row['title']}</b><br>Magnitude: {row['magnitude']}<br>Depth: {row['depth']} km"
    ).add_to(m)

m.save("earthquake_map.html")
print("✅ Mapa interactivo guardado en 'earthquake_map.html'")

"""

In [ ]:
# Preparar datos para el radar chart (top 10 terremotos)
df_top10 = df.nlargest(10, 'magnitude').reset_index(drop=True)

# -------------------------
# Helper functions
# -------------------------

def make_venn(d):
    from matplotlib_venn import venn3, venn2
    tsunami_set = set(d[d['tsunami']==1].index)
    red_alert_set = set(d[d['alert']==traduccion_colores['red']].index) if 'alert' in d.columns else set()
    mag6_set = set(d[d['magnitude']>6].index)
    fig, ax = plt.subplots(figsize=(4,4))
    
    # TODO if any of the sets is empty or sets are duplicates, treat it accordingly
    sets = [red_alert_set, tsunami_set, mag6_set]
    labels = ['Alerta Roja', 'Tsunami', 'Magnitud>6']
    
    # Filter out empty sets
    non_empty = [(s, l) for s, l in zip(sets, labels) if len(s) > 0]
    
    if len(non_empty) == 0:
        # All sets are empty
        ax.text(0.5, 0.5, 'No hay datos para mostrar', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    elif len(non_empty) == 1:
        # Only one non-empty set
        ax.text(0.5, 0.5, f'{non_empty[0][1]}: {len(non_empty[0][0])} terremotos', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    elif len(non_empty) == 2:
        # Two non-empty sets - use venn2
        s1, s2 = non_empty[0][0], non_empty[1][0]
        v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                  set_labels=(non_empty[0][1], non_empty[1][1]), ax=ax)
        # Hide zero labels
        for text in v.subset_labels:
            if text is not None and text.get_text() == '0':
                text.set_visible(False)
    else:
        # Three non-empty sets - check for duplicates
        if red_alert_set == tsunami_set == mag6_set:
            # All three sets are identical
            ax.text(0.5, 0.5, f'Todos los conjuntos son idénticos\n{len(red_alert_set)} terremotos', 
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis('off')
        elif red_alert_set == tsunami_set or red_alert_set == mag6_set or tsunami_set == mag6_set:
            # Two sets are identical - use venn2 with the unique ones
            if red_alert_set == tsunami_set:
                s1, s2 = red_alert_set, mag6_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Alerta Roja & Tsunami', 'Magnitud>6'), ax=ax)
            elif red_alert_set == mag6_set:
                s1, s2 = red_alert_set, tsunami_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Alerta Roja & Magnitud>6', 'Tsunami'), ax=ax)
            else:  # tsunami_set == mag6_set
                s1, s2 = red_alert_set, tsunami_set
                v = venn2(subsets=(len(s1 - s2), len(s2 - s1), len(s1 & s2)),
                          set_labels=('Alerta Roja', 'Tsunami & Magnitud>6'), ax=ax)
            # Hide zero labels
            for text in v.subset_labels:
                if text is not None and text.get_text() == '0':
                    text.set_visible(False)
        else:
            # All three sets are different - calculate subset sizes explicitly
            only_red = len(red_alert_set - tsunami_set - mag6_set)
            only_tsunami = len(tsunami_set - red_alert_set - mag6_set)
            red_and_tsunami = len((red_alert_set & tsunami_set) - mag6_set)
            only_mag6 = len(mag6_set - red_alert_set - tsunami_set)
            red_and_mag6 = len((red_alert_set & mag6_set) - tsunami_set)
            tsunami_and_mag6 = len((tsunami_set & mag6_set) - red_alert_set)
            all_three = len(red_alert_set & tsunami_set & mag6_set)
            
            v = venn3(subsets=(only_red, only_tsunami, red_and_tsunami, only_mag6, 
                              red_and_mag6, tsunami_and_mag6, all_three),
                      set_labels=('Alerta Roja', 'Tsunami', 'Magnitud>6'), ax=ax)
            # Hide zero labels
            for text in v.subset_labels:
                if text is not None and text.get_text() == '0':
                    text.set_visible(False)
    
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', transparent=True)
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return f'data:image/png;base64,{encoded}'

def filter_df(continent, country, year_range, only_tsunami, alerts_selected):
    d = df.copy()
    if continent and continent != 'All':
        d = d[d['continent'] == continent]
    if country and country != 'All':
        d = d[d['country'] == country]
    if year_range:
        start, end = year_range
        d = d[(d['year'] >= start) & (d['year'] <= end)]
    if only_tsunami:
        d = d[d['tsunami'] == 1]
    if alerts_selected and 'All' not in alerts_selected:
        d = d[d['alert'].isin(alerts_selected)]
    return d


# KPI cards generator
def kpi_card(title, value, subtitle=""):
    return dbc.Card(
        dbc.CardBody([
            html.H6(title, className="card-title text-muted"),
            html.H4(value, className="card-value"),
            html.Div(subtitle, className="card-subtitle text-muted")
        ]), className="h-100"
    )

# -------------------------
# App Layout
# -------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server

sidebar = dbc.Card([
    html.H5("Filtros", className="mt-2 mb-2"),
    dbc.Label("Continente"),
    dcc.Dropdown(options=['All'] + CONTINENTS, value='All', id='filter-continent', clearable=False),
    dbc.Label("País", className='mt-2'),
    dcc.Dropdown(options=['All'] + COUNTRIES, value='All', id='filter-country', clearable=False),
    dbc.Label("Años", className='mt-2'),
    dcc.RangeSlider(
        id='filter-year',
        min=YEAR_MIN,
        max=YEAR_MAX,
        step=1,
        value=[YEAR_MIN, YEAR_MAX],
        marks={YEAR_MIN: str(YEAR_MIN), YEAR_MAX: str(YEAR_MAX)}
    ),
    html.Div(id='year-selected', className='mt-2 text-center'),
    dbc.Checklist(options=[{"label": "Solo tsunamis", "value": "tsunami"}], value=[], id='filter-tsunami', inline=True, className='mt-3'),
    dbc.Label("Alertas", className='mt-2'),
    dcc.Dropdown(options=ALERTS, value=ALERTS, id='filter-alerts', multi=True),
    dbc.Button("Aplicar filtros", id='apply-filters', color='primary', className='me-3 mt-3', n_clicks=0),
    dbc.Button("Reset", id='reset-filters', color='secondary', className='mt-3', n_clicks=0)
], body=True)

# Add this before your layout definition or at the top with other components

footer = dbc.Container([
    html.Hr(className='my-4'),
    dbc.Row([
        dbc.Col([
            html.H5("Sobre el Dashboard", className='mb-3'),
            html.P([
                "Dashboard interactivo de análisis de terremotos a nivel mundial. ",
                "Desarrollado para visualizar y explorar datos sísmicos de manera intuitiva."
            ]),
            html.P([
                html.Strong("Autor: "),
                "Juan Arturo Abaurrea Calafell",
                html.Br(),
                html.Strong("Contacto: "),
                html.A("LinkedIn", href="https://www.linkedin.com/in/juan-arturo-abaurrea-calafell-238797225", target="_blank"),
                " | ",
                html.A("GitHub", href="https://github.com/Jarturog", target="_blank")
            ])
        ], width=12, md=4),
        
        dbc.Col([
            html.H5("Fuente de Datos", className='mb-3'),
            html.P([
                html.Strong("Dataset: "),
                "Earthquake dataset" # kaggle
            ]),
            html.P([
                "Los datos provienen de ",
                html.A("Kaggle",  # kaggle
                       href="https://www.kaggle.com/datasets/warcoder/earthquake-dataset/data",
                       target="_blank"),
                "."
            ]),
            html.P([
                html.Small("Última actualización: Octubre 2025", className='text-muted')
            ])
        ], width=12, md=4),
        
        dbc.Col([
            html.H5("Tecnologías", className='mb-3'),
            html.Ul([
                html.Li([html.Strong("Python"), " - Lenguaje de programación"]),
                html.Li([html.Strong("Dash & Plotly"), " - Framework de visualización"]),
                html.Li([html.Strong("Pandas"), " - Procesamiento de datos"]),
                html.Li([html.Strong("Bootstrap"), " - Diseño responsive"]),
                html.Li([html.Strong("TODO"), " - Añadir el resto de tecnologías"]),
            ], className='mb-0')
        ], width=12, md=4)
    ]),
    html.Hr(className='my-3'),
    dbc.Row([
        dbc.Col([
            html.P([
                "2025 - Dashboard de Terremotos | ",
                html.Small("Desarrollado para la asignatura Visualización Avanzada de Datos del Máster Universitario en Aprendizaje Automático y Datos Masivos de la Universidad Politécnica de Madrid")
            ], className='text-center text-muted mb-0')
        ])
    ])
], fluid=True, className='mt-5 mb-3')

# Update your layout to include the footer at the end:

layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Dashboard de Terremotos"), width=10)
    ], align='center', justify='center', className='mb-3 mt-4'),

    dbc.Row([
        dbc.Col([
            sidebar,
            html.Div(id='kpi-col', className='mt-3')
        ], width=2),

        dbc.Col([
            dcc.Tabs(id='tabs', value='tab-general', children=[
                dcc.Tab(label='General', value='tab-general'),
                dcc.Tab(label='Espacio', value='tab-espacio'),
                dcc.Tab(label='Tiempo', value='tab-tiempo'),
            ]),
            html.Div(id='tab-content', className='mt-3')
        ], width=8),
    ], justify='center'),
    
    # Add footer here
    footer
    
], style={'background': "#f0f0f4"}, fluid=True)

app.layout = layout

estilo_columna = {
    'backgroundColor': '#ffffff',
    'padding': '0.5em 1em',
    'borderRadius': '0.5em',
    'height': '30rem'
}
# -------------------------
# Callbacks: KPIs & Tab content
# -------------------------
@app.callback(
    Output('year-selected', 'children'),
    Input('filter-year', 'value')
)
def update_year_label(selected_range):
    return f"{selected_range[0]} - {selected_range[1]}"

@app.callback(
    Output('kpi-col', 'children'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_kpis(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    total = len(d)
    
    # Cálculos básicos
    avg_mag = round(d['magnitude'].mean(), 2) if total > 0 else 'N/A'
    max_mag = round(d['magnitude'].max(), 2) if total > 0 else 'N/A'
    min_mag = round(d['magnitude'].min(), 2) if total > 0 else 'N/A'
    
    # Tsunamis
    num_tsunamis = d['tsunami'].sum() if total > 0 and 'tsunami' in d.columns else 0
    pct_tsu = f"{round(100 * (num_tsunamis / total), 1)}%" if total > 0 else '0%'
    
    # Profundidad
    avg_depth = round(d['depth'].mean(), 1) if total > 0 and 'depth' in d.columns else 'N/A'
    max_depth = round(d['depth'].max(), 1) if total > 0 and 'depth' in d.columns else 'N/A'
    
    # Significancia
    avg_sig = round(d['sig'].mean(), 0) if total > 0 and 'sig' in d.columns else 'N/A'
    max_sig = round(d['sig'].max(), 0) if total > 0 and 'sig' in d.columns else 'N/A'
    
    # Alertas
    # Alertas
    alertas_dist = d['alert'].value_counts().to_dict() if 'alert' in d.columns else {}
    num_rojas = alertas_dist.get('rojo', 0)
    num_naranjas = alertas_dist.get('naranja', 0)
    num_amarillas = alertas_dist.get('amarillo', 0)
    num_verdes = alertas_dist.get('verde', 0)

    # Alertas críticas: rojas + naranjas
    pct_criticas = f"{round(100 * (num_rojas + num_naranjas) / total, 1)}%" if total > 0 else '0%'

    # Alertas moderadas: fusionar amarillas y verdes
    num_moderadas = num_amarillas + num_verdes
    pct_moderadas = f"{round(100 * num_moderadas / total, 1)}%" if total > 0 else '0%'

    # Nueva estadística: Terremotos de magnitud >= 7
    num_mag7 = len(d[d['magnitude'] >= 7]) if total > 0 else 0
    pct_mag7 = f"{round(100 * num_mag7 / total, 1)}%" if total > 0 else '0%'

    # Intensidades
    avg_mmi = round(d['mmi'].mean(), 1) if total > 0 and 'mmi' in d.columns and d['mmi'].notna().any() else 'N/A'
    avg_cdi = round(d['cdi'].mean(), 1) if total > 0 and 'cdi' in d.columns and d['cdi'].notna().any() else 'N/A'
    
    # Países más afectados
    if 'country' in d.columns and total > 0:
        top_country = d[d['country'] != VALOR_SUSTITUTO_NULO]['country'].value_counts().head(1)
        if len(top_country) > 0:
            pais_mas_afectado = f"{top_country.index[0]} ({top_country.values[0]})"
        else:
            pais_mas_afectado = 'N/A'
    else:
        pais_mas_afectado = 'N/A'
    
    # Continentes más afectados
    if 'continent' in d.columns and total > 0:
        top_continent = d[d['continent'] != VALOR_SUSTITUTO_NULO]['continent'].value_counts().head(1)
        if len(top_continent) > 0:
            continente_mas_afectado = f"{top_continent.index[0]} ({top_continent.values[0]})"
        else:
            continente_mas_afectado = 'N/A'
    else:
        continente_mas_afectado = 'N/A'
    
    # Tendencia temporal (si hay datos de diferentes años)
    if 'year' in d.columns and total > 0:
        years_data = d.groupby('year').size()
        if len(years_data) > 1:
            trend = "↑" if years_data.iloc[-1] > years_data.iloc[0] else "↓"
            trend_text = f"{trend} {abs(round(((years_data.iloc[-1] - years_data.iloc[0]) / years_data.iloc[0]) * 100, 1))}%"
        else:
            trend_text = "N/A"
    else:
        trend_text = "N/A"
    
    # Terremoto más significativo
    if total > 0:
        most_sig = d.nlargest(1, 'magnitude').iloc[0]
        terremoto_principal = f"{most_sig['title']}"
    else:
        terremoto_principal = 'N/A'

    cards = dbc.Row([
        # Fila 1: Información general
        dbc.Col(kpi_card('Total Terremotos', total, [f'Tendencia:', html.Br(), f'{trend_text}']), width=6, className='mb-2'),
        dbc.Col(kpi_card('Magnitud Media', avg_mag, [f'Rango:', html.Br(), f'{min_mag} - {max_mag}']), width=6, className='mb-2'),

        dbc.Col(kpi_card('Tsunamis', [f'{num_tsunamis}', html.Br(), f'({pct_tsu})'], 'del total de eventos'), width=6, className='mb-2'),
        dbc.Col(kpi_card('Terremotos ≥ M7', f'{num_mag7}', f'({pct_mag7})'), width=6, className='mb-2'),
        
        dbc.Col(kpi_card('Alertas Críticas', pct_criticas, [f'🔴 {num_rojas}', html.Br(), f'🟠 {num_naranjas}']), width=6, className='mb-2'),
        dbc.Col(kpi_card('Alertas Moderadas', pct_moderadas, [f'🟡 {num_amarillas}', html.Br(), f'🟢 {num_verdes}']), width=6, className='mb-2'),

        dbc.Col(kpi_card('Profundidad Media', f'{avg_depth} km', [f'Máxima:', html.Br(), f'{max_depth} km']), width=6, className='mb-2'),
        dbc.Col(kpi_card('Significancia Media', avg_sig, [f'Máxima:', html.Br(), f'{max_sig}']), width=6, className='mb-2'),
        
        dbc.Col(kpi_card('MMI Promedio', avg_mmi, 'Intensidad instrumental'), width=6, className='mb-2'),
        dbc.Col(kpi_card('CDI Promedio', avg_cdi, 'Intensidad reportada'), width=6, className='mb-2'),
        
        dbc.Col(kpi_card('País Más Afectado', pais_mas_afectado, ''), width=6, className='mb-2'),
        dbc.Col(kpi_card('Continente Más Afectado', continente_mas_afectado, ''), width=6, className='mb-2'),
        
        dbc.Col(kpi_card('Terremoto Con Mayor Magnitud', terremoto_principal, f'Magnitud: {max_mag}'), width=12, className='mb-2'),
    ], className='g-2 align-items-stretch')

    return cards


@app.callback(
    Output('tab-content', 'children'),
    Input('tabs', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def render_tab(tab, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)

    if tab == 'tab-espacio':
        # Map + choropleth + small table
        map_fig = px.scatter_geo(
            d.dropna(subset=['latitude','longitude']),
            lat='latitude', lon='longitude',
            hover_name='title',
            size='magnitude',
            color='alert' if 'alert' in d.columns else None,
            title='Mapa de epicentros',
            projection='natural earth'
        )
        map_fig.update_layout(height=600)

        choropleth = None
        if 'country' in d.columns:
            country_agg = d.groupby('country').agg({'magnitude':'mean','title':'count'}).reset_index().rename(columns={'title':'count'})
            # we rely on country names; for robust choropleth a mapping to ISO codes is recommended
            choropleth = px.bar(country_agg.sort_values('count', ascending=False).head(12), x='country', y='count', title='Top países por número de terremotos')
            choropleth.update_layout(height=350)

        return html.Div([
            dbc.Row([
                dbc.Col(dcc.Graph(figure=map_fig), width=8),
                dbc.Col(dcc.Graph(figure=choropleth) if choropleth is not None else html.Div(), width=4)
            ]),
            dbc.Row([
                dbc.Col(html.Div([html.H5('Distribución por alertas'), dcc.Graph(figure=px.pie(d, names='alert', title='Alertas', color_discrete_map=colores_alerta))]), width=4),
                dbc.Col(html.Div([html.H5('Magnitud vs Profundidad'), dcc.Graph(figure=px.scatter(d, x='magnitude', y='depth', title='Magnitud vs Profundidad', hover_data=['title']) )]), width=8)
            ])
        ])

    elif tab == 'tab-tiempo':
        # Time series + calendar heatmap approximation + animated frames by year
        ts = d.dropna(subset=['date_time'])
        if len(ts) == 0:
            return html.Div(html.H4('No hay datos para este rango y filtros'))

        mag_time = ts.set_index('date_time').resample('M').magnitude.mean().reset_index()
        fig_ts = px.line(mag_time, x='date_time', y='magnitude', title='Magnitud promedio por mes')
        fig_ts.update_layout(height=450)

        # animation by year: scatter_geo with animation_frame
        if 'year' in ts.columns:
            anim = px.scatter_geo(ts.dropna(subset=['latitude','longitude']), lat='latitude', lon='longitude', color='alert' if 'alert' in ts.columns else None, size='magnitude', animation_frame='year', hover_name='title', projection='natural earth', title='Animación por año')
            anim.update_layout(height=500)
        else:
            anim = go.Figure()

        return html.Div([
            dbc.Row([dbc.Col(dcc.Graph(figure=fig_ts), width=6), dbc.Col(dcc.Graph(figure=anim), width=6)]),
            dbc.Row([dbc.Col(html.Div([html.H5('Calendar-like heatmap (por día)'), dcc.Graph(figure=calendar_heatmap(ts))]), width=12)])
        ])

    elif tab == 'tab-general':
        radar_fig = radar_placeholder()

        return html.Div([
            # PRIMERA FILA: Violin, Density, Pie/Waffle (las 3 distribuciones)
            dbc.Row([
                dbc.Col(html.Div([
                    html.H5('Distribución de Magnitud por: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                    dcc.RadioItems(
                        id='violin-groupby',
                        options=[
                            {'label': 'Alerta', 'value': 'alert'},
                            {'label': 'Tipo de Magnitud', 'value': 'magType'}
                        ],
                        value='alert',
                        inline=True,
                        labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                        className='mb-0'
                    ),
                    dcc.Graph(id='violin-graph')
                ], style=estilo_columna), width=4),
                dbc.Col(html.Div([
                    html.H5('Distribución de: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                    dcc.RadioItems(
                        id='density-variable',
                        options=[{'label': 'Magnitud', 'value': 'magnitude'}, {'label': 'Profundidad', 'value': 'depth'}],
                        value='magnitude',
                        inline=True,
                        labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                        className='mb-0'
                    ),
                    dcc.Graph(id='density-graph')
                ], style=estilo_columna), width=4),
                dbc.Col(html.Div([
                    html.H5('Distribución de: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                    dcc.RadioItems(
                        id='pie-variable',
                        options=[
                            {'label': 'Alertas', 'value': 'alert'},
                            {'label': 'Tipos de Cálculo de Magnitud', 'value': 'magType'}
                        ],
                        value='alert',
                        inline=True,
                        labelStyle={'margin-right': '1.5em', 'whiteSpace': 'nowrap'},
                        className='mb-0'
                    ),
                    dcc.Dropdown(
                        id='pie-chart-type',
                        options=[{'label': 'Pie Chart', 'value': 'pie'}, {'label': 'Waffle Chart', 'value': 'waffle'}],
                        value='pie',
                        clearable=False,
                        className='mb-2'
                    ),
                    dcc.Graph(id='pie-waffle-graph')
                ], style=estilo_columna), width=4)
            ], className='mb-3'),
            dbc.Row([
                dbc.Col(html.Div([
                    html.Div([
                        html.H5('Matriz de Correlación: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                        dcc.Dropdown(
                            id='heatmap-vars',
                            options=[
                                {'label': 'Todas las variables', 'value': 'all'},
                                {'label': 'Magnitud y relacionadas', 'value': 'magnitude_related'},
                                {'label': 'Geográficas', 'value': 'geographic'}
                            ],
                            value='all',
                            clearable=False,
                            className='mb-0',
                            style={'minWidth': '200px'}
                        ),
                    ], style={'display': 'flex', 'alignItems': 'center'}),
                    dcc.Graph(id='heatmap-graph')
                ], style=estilo_columna), width=8),
                dbc.Col(html.Div([
                    html.H5('Diagrama de Venn', className='mb-2'),
                    html.Img(id='venn-diagram', style={'width': '100%', 'height': 'auto'})
                ], style=estilo_columna), width=4)
            ], className='mb-3'),
            dbc.Row([
                dbc.Col(html.Div([
                    html.Div([
                        html.H5('Bubble Plot: ', className='me-3 mb-0', style={'whiteSpace': 'nowrap'}),
                        dcc.Dropdown(
                            id='bubble-axes',
                            options=[
                                {'label': 'Magnitud vs Profundidad', 'value': 'mag_depth'},
                                {'label': 'Magnitud vs Significancia', 'value': 'mag_sig'},
                                {'label': 'Profundidad vs Significancia', 'value': 'depth_sig'}
                            ],
                            value='mag_depth',
                            clearable=False,
                            className='mb-0',
                            style={'width': '75%'}
                        ),
                    ], style={'display': 'flex', 'alignItems': 'center'}),
                    dcc.Graph(id='bubble-graph')
                ], style=estilo_columna), width=7),
                dbc.Col(html.Div([
                    html.H5('Radar del Top 10 Terremotos'), 
                    dcc.Dropdown(
                        id='radar-select',
                        options=[{'label': r, 'value': i} for i, r in enumerate(df_top10['title'])],
                        value=0,
                        style={'whiteSpace': 'nowrap'}
                    ), 
                    dcc.Graph(id='radar-graph', figure=radar_fig)
                ], style=estilo_columna), width=5)
            ], className='mb-3')
        ])
    else:
        return html.Div(html.H4('Pestaña no encontrada'))

# -------------------------
# Extra helper plots (calendar heatmap & radar)
# -------------------------

# Callback for Violin chart
@app.callback(
    Output('violin-graph', 'figure'),
    Input('violin-groupby', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_violin(groupby_var, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if groupby_var not in d.columns or len(d) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"No hay datos suficientes", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    fig = px.violin(
        d,
        y='magnitude',
        x=groupby_var,
        box=True,
        points=None,
        color=groupby_var,
        color_discrete_map=colores_alerta if groupby_var == 'alert' else None,
        category_orders={'alert': ALERTS} if groupby_var == 'alert' else None
    )
    fig.update_layout(
        height=350,
        xaxis_title=groupby_var.capitalize(),
        yaxis_title='Magnitud',
        showlegend=True,
        legend=dict(
            orientation="h",  # Leyenda horizontal
            yanchor="bottom",
            y=1.02,  # Posición encima del gráfico
            xanchor="center",
            x=0.5
        ),
        margin=dict(l=50, r=20, t=40, b=50)  # Márgenes más ajustados
    )
    return fig

# Callback for Heatmap
@app.callback(
    Output('heatmap-graph', 'figure'),
    Input('heatmap-vars', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_heatmap(var_selection, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    numeric_cols = d.select_dtypes(include='number').columns.tolist()
    
    if var_selection == 'magnitude_related':
        cols = [c for c in ['magnitude', 'sig', 'cdi', 'mmi', 'dmin', 'gap'] if c in numeric_cols]
    elif var_selection == 'geographic':
        cols = [c for c in ['latitude', 'longitude', 'depth'] if c in numeric_cols]
    else:  # 'all'
        cols = numeric_cols
    
    if len(cols) < 2:
        fig = go.Figure()
        fig.add_annotation(text="No hay suficientes variables numéricas", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    corr_matrix = d[cols].corr()
    
    fig = px.imshow(
        corr_matrix,
        text_auto='.2f',
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1
    )
    fig.update_layout(height=350)
    return fig


# Callback for Bubble/Scatter chart
@app.callback(
    Output('bubble-graph', 'figure'),
    Input('bubble-axes', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_bubble(axes_selection, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    axes_map = {
        'mag_depth': ('magnitude', 'depth', 'Magnitud vs Profundidad'),
        'mag_sig': ('magnitude', 'sig', 'Magnitud vs Significancia'),
        'depth_sig': ('depth', 'sig', 'Profundidad vs Significancia')
    }
    
    x_var, y_var, title = axes_map.get(axes_selection, ('magnitude', 'depth', 'Magnitud vs Profundidad'))
    
    if x_var not in d.columns or y_var not in d.columns:
        fig = go.Figure()
        fig.add_annotation(text=f"Variables no disponibles", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    size = None
    if 'mag' != x_var and 'mag' != y_var:
        size = 'magnitude'
    elif 'depth' != x_var and 'depth' != y_var:
        size = 'depth'
    elif 'sig' != x_var and 'sig' != y_var:
        size = 'sig'
    
    if size in d.columns:
        d['size_scaled'] = d[size] ** 10 / 10000000
    else:
        d['size_scaled'] = None

    fig = px.scatter(
        d.dropna(subset=[x_var, y_var]),
        x=x_var,
        y=y_var,
        size='size_scaled' if size in d.columns else None,
        color='alert' if 'alert' in d.columns else None,
        hover_data=['title'],
        category_orders={'alert': ALERTS} if 'alert' in d.columns else None,
        color_discrete_map=colores_alerta if 'alert' in d.columns else None
    )
    fig.update_layout(
        height=350,
        xaxis_title=x_var.capitalize(),
        yaxis_title=y_var.capitalize()
    )
    return fig


# Update Venn diagram callback to be reactive to filters
@app.callback(
    Output('venn-diagram', 'src'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_venn(n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    return make_venn(d)

def calendar_heatmap(ts_df):
    # Build a monthly heatmap (year x month) with counts to approximate calendar heatmap
    dfc = ts_df.copy()
    dfc['year_month'] = dfc['date_time'].dt.to_period('M')
    counts = dfc.groupby('year_month').size().reset_index(name='count')
    if counts.empty:
        return go.Figure()
    counts['year'] = counts['year_month'].dt.year
    counts['month'] = counts['year_month'].dt.month
    pivot = counts.pivot(index='year', columns='month', values='count').fillna(0)
    fig = go.Figure(data=go.Heatmap(z=pivot.values, x=list(pivot.columns), y=list(pivot.index), colorscale='Blues'))
    fig.update_layout(title='Terremotos por mes (heatmap)', xaxis_title='Mes', yaxis_title='Año', height=400)
    return fig


def radar_placeholder():
    # default empty radar
    fig = go.Figure()
    fig.update_layout(height=300, title='Selecciona un terremoto para ver radar')
    return fig

# Radar update callback
@app.callback(
    Output('radar-graph', 'figure'),
    Input('radar-select', 'value')
)
def update_radar(selected_idx):
    if selected_idx is None or selected_idx >= len(df_top10):
        return radar_placeholder()
    eq = df_top10.iloc[selected_idx]
    variables = ['magnitude', 'depth', 'sig', 'cdi', 'mmi']
    labels = ['Magnitud','Profundidad','Significancia','CDI','MMI']
    values = []
    for v in variables:
        if v in df.columns and pd.notna(eq.get(v)):
            vmin, vmax = df[v].min(), df[v].max()
            if pd.isna(vmin) or pd.isna(vmax) or vmax==vmin:
                values.append(50)
            else:
                values.append((eq[v]-vmin)/(vmax-vmin)*100)
        else:
            values.append(0)
    values.append(values[0])
    labels_closed = labels + [labels[0]]
    
    fig = go.Figure(go.Scatterpolar(
        r=values,
        theta=labels_closed,
        fill='toself',
        name=eq.get('title','evento'),
        hovertemplate='%{theta}: %{r:.2f}%'
    ))
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                range=[0,100],
                showticklabels=False
            )
        ), 
        height=350
    )
    return fig

@app.callback(
    Output('density-graph', 'figure'),
    Input('density-variable', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_density(variable, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    import numpy as np
    from scipy.stats import gaussian_kde

    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    # Clean data
    x = d[variable].dropna()
    if len(x) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"No hay datos para {variable}", x=0.5, y=0.5, showarrow=False)
        return fig

    # Create histogram
    hist = go.Histogram(
        x=x,
        histnorm='probability density',
        nbinsx=40,
        name='Histograma',
        opacity=0.5,
        marker_color='lightblue'
    )

    # Create KDE line
    kde = gaussian_kde(x)
    xs = np.linspace(x.min(), x.max(), 300)
    ys = kde(xs)
    line = go.Scatter(
        x=xs,
        y=ys,
        mode='lines',
        line=dict(color='royalblue', width=2.5),
        name='Densidad (KDE)'
    )

    # Combine
    fig = go.Figure(data=[hist, line])
    fig.update_layout(
        xaxis_title=variable.capitalize(),
        yaxis_title='Densidad',
        height=350,
        template='plotly_white',
        legend=dict(
            orientation="h",  # Leyenda horizontal
            yanchor="bottom",
            y=1.02,  # Posición encima del gráfico
            xanchor="center",
            x=0.5
        ),
        margin=dict(l=50, r=20, t=40, b=50)  # Márgenes más ajustados
    )
    return fig
@app.callback(
    Output('pie-waffle-graph', 'figure'),
    Input('pie-variable', 'value'),
    Input('pie-chart-type', 'value'),
    Input('apply-filters', 'n_clicks'),
    State('filter-continent', 'value'),
    State('filter-country', 'value'),
    State('filter-year', 'value'),
    State('filter-tsunami', 'value'),
    State('filter-alerts', 'value')
)
def update_pie_waffle(variable, chart_type, n_clicks, continent, country, year_range, tsunami_flag, alerts_selected):
    only_tsu = 'tsunami' in tsunami_flag if tsunami_flag is not None else False
    d = filter_df(continent, country, year_range, only_tsu, alerts_selected)
    
    if variable not in d.columns:
        fig = go.Figure()
        fig.add_annotation(text=f"Column '{variable}' not found", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        return fig
    
    # Remove NaN values for the selected variable
    d_clean = d[d[variable].notna()].copy()
    
    if len(d_clean) == 0:
        fig = go.Figure()
        fig.add_annotation(text=f"No data available for {variable}", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)
        return fig
    
    if chart_type == 'pie':
        fig = px.pie(
            d_clean,
            names=variable,
            color=variable,
            color_discrete_map=colores_alerta if variable == 'alert' else None,
            category_orders={'alert': ALERTS} if variable == 'alert' else None
        )
        fig.update_layout(height=350)
    elif chart_type == 'waffle':
        # Prepare data
        counts = d_clean[variable].value_counts().to_dict()
        counts = {k: v for k, v in counts.items() if v >= 10}
        other_count = len(d_clean) - sum(counts.values())
        if other_count > 0:
            counts['other'] = other_count
        # Create matplotlib figure with pywaffle
        fig_mpl = plt.figure(
            FigureClass=Waffle,
            rows=10,
            columns=10,
            values=counts,
            labels=[f"{k}: {v}" for k, v in counts.items()],
            colors=[colores_alerta.get(k, None) for k in counts.keys()] if variable == 'alert' else None,
            legend={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
            figsize=(8, 5)
        )
        
        # Convert matplotlib figure to plotly
        buf = io.BytesIO()
        fig_mpl.savefig(buf, format='png', bbox_inches='tight', dpi=100)
        buf.seek(0)
        encoded = base64.b64encode(buf.read()).decode()
        plt.close(fig_mpl)
        
        # Create plotly figure with the image
        fig = go.Figure()
        fig.add_layout_image(
            dict(
                source=f'data:image/png;base64,{encoded}',
                xref="paper", yref="paper",
                x=0, y=1,
                sizex=1, sizey=1,
                xanchor="left", yanchor="top",
                sizing="contain",
                layer="below"
            )
        )
        fig.update_xaxes(visible=False, range=[0, 1])
        fig.update_yaxes(visible=False, range=[0, 1])
        fig.update_layout(
            height=350,
            margin=dict(l=0, r=0, t=0, b=0),
            plot_bgcolor='white'
        )
    else:
        fig = go.Figure()
        fig.add_annotation(text="Tipo de gráfico no soportado", xref="paper", yref="paper", x=0.5, y=0.5, showarrow=False)
        fig.update_layout(height=350)

    return fig

# -------------------------
# Reset filters callback
# -------------------------
@app.callback(
    [Output('filter-continent', 'value'), Output('filter-country', 'value'), Output('filter-year', 'value'), Output('filter-tsunami', 'value'), Output('filter-alerts', 'value')],
    Input('reset-filters', 'n_clicks')
)
def reset_filters(n):
    return 'All', 'All', [YEAR_MIN, YEAR_MAX], [], ['All']

# -------------------------
# Run
# -------------------------

if __name__ == '__main__':
    app.run(debug=True, port=8050)
